In [ ]:
import requests

# 1. Internal cluster service URL for RHOAI 2.25
BASE_URL = "http://redhattraining-model-registry.rhoai-model-registries.svc.cluster.local:8080/api/model_registry/v1alpha3"
HEADERS = {"Content-Type": "application/json"}

print(f"Targeting Registry Endpoint: {BASE_URL}")

In [ ]:
def register_model_via_rest(item):
    print(f"\nProcessing: {item['name']} ({item['version']})...")
    
    # Step A: Post or Get Parent Registered Model Container
    model_payload = {"name": item["name"], "state": "LIVE"}
    res = requests.post(f"{BASE_URL}/registered_models", json=model_payload, headers=HEADERS)
    
    if res.status_code == 201:
        model_id = res.json().get("id")
        print(f"  ➡ Created Registered Model ID: {model_id}")
    elif res.status_code == 409:
        get_res = requests.get(f"{BASE_URL}/registered_models", headers=HEADERS)
        model_id = next((m["id"] for m in get_res.json().get("items", []) if m["name"] == item["name"]), None)
        print(f"  ➡ Model container already exists. Linked to ID: {model_id}")
    else:
        print(f"  ❌ Failed model container check: {res.text}")
        return

    # Step B: Establish the unique Version Track
    version_payload = {"name": item["version"], "registeredModelId": model_id, "state": "LIVE"}
    v_res = requests.post(f"{BASE_URL}/model_versions", json=version_payload, headers=HEADERS)
    
    if v_res.status_code == 201:
        version_id = v_res.json().get("id")
        print(f"  ➡ Created Version ID: {version_id}")
    elif v_res.status_code == 409:
        # Since the version exists but the artifact failed previously, let's fetch the version ID to fix the artifact
        get_v_res = requests.get(f"{BASE_URL}/registered_models/{model_id}/versions", headers=HEADERS)
        version_id = next((v["id"] for v in get_v_res.json().get("items", []) if v["name"] == item["version"]), None)
        print(f"  ➡ Version already exists. Repairing artifact links for Version ID: {version_id}")
    else:
        print(f"  ❌ Failed version declaration: {v_res.text}")
        return

    # Step C: Attach the OCI ModelCar Artifact parameters
    artifact_payload = {
        "name": f"{item['name']}-oci-artifact",
        "uri": item["uri"],
        "state": "LIVE",
        "modelArtifactType": "ModelArtifact",
        "customProperties": {
            "model_format_name": {"string_value": "onnx"},
            "model_format_version": {"string_value": "1"}
        }
    }
    a_res = requests.post(f"{BASE_URL}/model_versions/{version_id}/artifacts", json=artifact_payload, headers=HEADERS)
    
    if a_res.status_code == 201:
        print(f"  ✅ Success! OCI link verified.")
    else:
        print(f"  ❌ Failed artifact mapping: {a_res.text}")

In [ ]:
# Modifiable inventory list
models_to_import = [
    {"name": "diabetes-1", "version": "v2.25.1", "uri": "oci://registry.ocp4.example.com:8443/redhattraining/models/diabetes:2.25.1"},
    {"name": "diabetes-2", "version": "v2.25.2", "uri": "oci://registry.ocp4.example.com:8443/redhattraining/models/diabetes:2.25.2"},
    {"name": "diabetes-3", "version": "v2.25.3", "uri": "oci://registry.ocp4.example.com:8443/redhattraining/models/diabetes:2.25.3"},
    {"name": "diabetes-4", "version": "v2.25.4", "uri": "oci://registry.ocp4.example.com:8443/redhattraining/models/diabetes:2.25.4"},
    {"name": "diabetes-5", "version": "v2.25.5", "uri": "oci://registry.ocp4.example.com:8443/redhattraining/models/diabetes:2.25.5"}
]

# Run loop against your inventory
for model_item in models_to_import:
    register_model_via_rest(model_item)

print("\n----------------------------------------")
print("Bulk ingestion run finished.")